# Mutate

The `.mutate()` method adds new columns or modifies existing columns in your
data. New columns are defined using keyword arguments, where the key becomes
the column name and the value is a Polars expression.

In [ ]:
import os, pathlib
_DATA_DIR = str(pathlib.Path(__import__('tidypolars_extra').__file__).parent / 'data')

import tidypolars_extra as tp

mtcars = tp.tibble(tp.read_data(fn=f"{_DATA_DIR}/mtcars.csv", sep=",", silently=True))

small_cars = mtcars.select("name", "cyl", "mpg", "hp")

## Assign new columns

Create a new column `cyl2` that doubles the `cyl` value:

In [ ]:
small_cars.mutate(cyl2=tp.col("cyl") * 2)

You can also add a column with a scalar value:

In [ ]:
small_cars.mutate(label=tp.lit("car"))

## Combining expressions

You can define multiple columns in a single `mutate` call:

In [ ]:
small_cars.mutate(
    hp_per_cyl=tp.col("hp") / tp.col("cyl"),
    double_mpg=tp.col("mpg") * 2,
)

## Used with grouped operations

The `by` parameter lets you compute group-level statistics and attach them
back to each row. For example, computing the mean `hp` per `cyl` group and
then demeaning the values:

In [ ]:
(
    small_cars
    .mutate(
        hp_mean=tp.col("hp").mean(),
        demeaned_hp=tp.col("hp") - tp.col("hp").mean(),
        by="cyl",
    )
)

## With if_else and case_when

### Using `if_else`

`tp.if_else()` works like a ternary operator — it evaluates a condition and
returns one value when true and another when false:

In [ ]:
small_cars.mutate(
    hp_category=tp.if_else(tp.col("hp") > 150, tp.lit("high"), tp.lit("low"))
)

### Using `case_when`

`tp.case_when()` handles multiple conditions. Conditions are specified as
pairs of `(condition, value)`, with an optional `_default` for unmatched rows:

In [ ]:
small_cars.mutate(
    size=tp.case_when(
        tp.col("cyl") == 4, "small",
        tp.col("cyl") == 6, "medium",
        _default="large",
    )
)